# Fase 1 — Chain inicial da Auto Super Parts CWB

Somente a seção 8 da arquitetura: mensagem → templates → LLM → output parser → JSON validado.
A implementação é importada dos `.py`. Não há catálogo, SQL, ferramentas, memória, ViaCEP, embeddings ou RAG nesta fase.

As escolhas de intenções e pendências são **propostas**, registradas em `01_documentacao/registro_decisoes.csv`.


In [1]:
import os
import sys
from pathlib import Path

# Localização permite abrir o notebook pela raiz do repositório ou pela pasta de notebooks.
candidatos = [c for base in (Path.cwd(), *Path.cwd().parents) for c in (base, base / 'Agentes Cognitivos com LangChain' / 'Projeto final')]
PROJETO = next(c for c in candidatos if (c / '02_codigo/src/auto_super_parts').is_dir())
sys.path.insert(0, str(PROJETO / '02_codigo/src'))

from auto_super_parts.chain_inicial import criar_chains, criar_modelo, processar_mensagem
from auto_super_parts.prompts import criar_templates
from auto_super_parts.schemas import PedidoIdentificado
from auto_super_parts.demonstracao_fase01 import carregar_casos, criar_modelo_simulado, avaliar_casos, registrar_evidencia


## Ambiente e origem das respostas

O padrão é `real`: OpenAI, `gpt-4o-mini`. A chave é lida do ambiente ou do `.env` compartilhado da raiz e nunca é exibida.
`FASE01_MODO=simulado` seleciona respostas roteirizadas para testar a infraestrutura offline; **não testa interpretação da linguagem**. Não existe fallback automático de real para dummy.


In [2]:
from importlib.metadata import version

MODO = os.environ.get('FASE01_MODO', 'real')
assert MODO in ('real', 'simulado')
print('Modo:', MODO)
print('Python:', sys.version.split()[0])
print({p: version(p) for p in ('langchain-core', 'langchain-openai', 'pydantic', 'python-dotenv', 'nbformat', 'nbclient', 'ipykernel')})
caso = carregar_casos()[0]
mensagem = caso['mensagem']
print('Mensagem:', mensagem)


Modo: real
Python: 3.12.6
{'langchain-core': '1.7.0', 'langchain-openai': '1.7.0', 'pydantic': '2.14.0', 'python-dotenv': '1.2.4', 'nbformat': '5.11.1', 'nbclient': '0.11.0', 'ipykernel': '7.4.0'}
Mensagem: Tenho um Gol 2010 1.0 e quero duas velas


## 1. PromptTemplate e ChatPromptTemplate

Os dois templates descrevem a mesma tarefa PTCF. `PromptTemplate` produz uma string;
`ChatPromptTemplate` separa regras `system` do dado `human`. `partial()` fixa as instruções
do parser; `invoke()` preenche somente a mensagem atual. Nenhum template consulta o catálogo.


In [3]:
texto, chat, parser = criar_templates()
print('PROMPT TEMPLATE FORMATADO:\n')
print(texto.invoke({'mensagem': mensagem}).to_string())
print('\nCHAT TEMPLATE — papéis e trecho inicial de cada mensagem:')
for msg in chat.invoke({'mensagem': mensagem}).to_messages():
    print(msg.type, '->', msg.content[:220])


PROMPT TEMPLATE FORMATADO:

Persona: você é o atendente claro e cuidadoso da Auto Super Parts CWB, loja fictícia.
Tarefa: extrair intenção e entidades de UMA mensagem, sem executar atendimento ou dar recomendações.
Contexto: esta é apenas a fase 1, uma chain fixa. Não há catálogo consultável, memória, ferramenta,
API de CEP, cálculo de frete, RAG, preço ou estoque disponíveis. Não afirme compatibilidade real.
Não complete veículo, versão, combustível ou especificação de óleo por conhecimento geral.
Extraia somente dados explicitamente informados. Campos ausentes ficam null. Não assumir quantidade 1.
Não transforme ano em quantidade ou cilindrada em ano. Preserve código de peça se informado.
Normalize apenas plural inequívoco da peça (velas -> vela), números por extenso e hífen de CEP.
Se disser 1.0, registre apenas 1.0; não acrescente 8V. Não use modelos de veículos como fonte de combustível.
Se houver quantidade zero, negativa ou fracionária, preserve esse valor no JSON para o validad

## 2. Output parser e BaseModel

Teste **local**, sem chamada à LLM: uma saída roteirizada atravessa `PydanticOutputParser`.
O contrato usa `None` para ausentes e recalcula as pendências em Python. Para o exemplo,
combustível continua desconhecido; não acrescentamos `8V`, combustível ou compatibilidade.


In [4]:
import json

objeto = parser.invoke(json.dumps(caso['saida_simulada'], ensure_ascii=False))
print(type(objeto).__name__)
print(objeto.model_dump_json(indent=2))
assert objeto.veiculo.motor == '1.0'
assert objeto.veiculo.combustivel is None
assert objeto.quantidade == 2


PedidoIdentificado
{
  "intencao": "consulta_peca",
  "veiculo": {
    "modelo": "Gol",
    "ano_modelo": 2010,
    "motor": "1.0",
    "combustivel": null
  },
  "peca": "vela",
  "quantidade": 2,
  "cep": null,
  "campos_faltantes": [
    "veiculo.combustivel"
  ]
}


## 3. Erro tratado, sem apresentar JSON inválido como sucesso

A falha abaixo é **deliberadamente simulada**, não uma falha atribuída à OpenAI.
O modelo dummy devolve texto sem JSON; o mesmo tratamento usado pela CLI rejeita a saída.
Não há repetição automática nem transformação silenciosa de quantidade inválida em 1.


In [5]:
invalida = criar_chains(modelo=criar_modelo_simulado(['isto não é JSON']))
falha = processar_mensagem(invalida.lcel, 'Quero duas velas.')
print(falha.model_dump_json(indent=2))
assert falha.status == 'erro_validacao' and falha.dados is None


{
  "status": "erro_validacao",
  "dados": null,
  "erros": [
    "O modelo não devolveu JSON compatível com o contrato."
  ]
}


## 4. A mesma transformação com @chain e LCEL

O código é importado, não duplicado no notebook. Na composição `@chain`, as etapas são
invocadas explicitamente; LCEL declara a sequência com `|`. Adotamos chat + LCEL pela
separação de papéis e legibilidade do fluxo fixo. Isso é uma chain, sem autonomia de agente.


In [6]:
import inspect

print(inspect.getsource(criar_chains))
modelo = criar_modelo() if MODO == 'real' else criar_modelo_simulado([caso['saida_simulada']] * 2)
composicoes = criar_chains(modelo=modelo)
resultado_lcel = processar_mensagem(composicoes.lcel, mensagem)
resultado_chain = processar_mensagem(composicoes.decorada, mensagem)
print('LCEL:', resultado_lcel.model_dump_json(indent=2))
print('@chain:', resultado_chain.model_dump_json(indent=2))
print('Objetos iguais nesta execução:', resultado_lcel == resultado_chain)
# Mesmo contrato e etapas não são uma garantia de saídas idênticas em duas chamadas reais.


def criar_chains(modelo=None, template="chat"):
    if template not in ("chat", "texto"):
        raise ValueError("Template deve ser chat ou texto.")
    texto, chat, parser = criar_templates()
    prompt = chat if template == "chat" else texto
    if modelo is None:
        modelo = criar_modelo()
    # LCEL declara a sequência entrada -> prompt -> modelo -> parser, sem chamar a API na montagem.
    lcel = RunnableLambda(validar_entrada) | prompt | modelo | parser

    # @CHAIN demonstra a mesma transformação com controle explícito das etapas.
    # CONFIG é propagada para manter callbacks/tags de quem executar, sem adicionar ferramentas.
    @chain
    def decorada(entrada: dict, config: RunnableConfig):
        dados = validar_entrada(entrada)
        mensagens = prompt.invoke(dados, config=config)
        resposta = modelo.invoke(mensagens, config=config)
        return parser.invoke(resposta, config=config)

    return Composicoes(lcel=lcel, decorada=decorada, prompt=prompt, pars

LCEL: {
  "status": "ok",
  "dados": {
    "intencao": "consulta_peca",
    "veiculo": {
      "modelo": "Gol",
      "ano_modelo": 2010,
      "motor": "1.0",
      "combustivel": null
    },
    "peca": "vela",
    "quantidade": 2,
    "cep": null,
    "campos_faltantes": [
      "veiculo.combustivel"
    ]
  },
  "erros": []
}
@chain: {
  "status": "ok",
  "dados": {
    "intencao": "consulta_peca",
    "veiculo": {
      "modelo": "Gol",
      "ano_modelo": 2010,
      "motor": "1.0",
      "combustivel": null
    },
    "peca": "vela",
    "quantidade": 2,
    "cep": null,
    "campos_faltantes": [
      "veiculo.combustivel"
    ]
  },
  "erros": []
}
Objetos iguais nesta execução: True


## 5. batch com os casos de aceitação da fase 1

`avaliar_casos()` usa `.batch(..., return_exceptions=True)` na mesma chain, com
concorrência 1. Os casos são independentes. O gabarito permanece no Python, fora do prompt.
São verificados: exemplo do Gol, combustível explícito, filtro ambíguo, quantidade zero,
pagamento sem veículo, CEP com zero inicial, CEP inválido, intenção desconhecida e quantidade
faltante no orçamento. Falhas e divergências são salvas antes da asserção final.


In [7]:
resultados = avaliar_casos(MODO)
evidencia = registrar_evidencia(resultados, MODO)
for registro in resultados:
    print('\n', registro['id'], registro['mensagem'])
    print(json.dumps(registro['observado'], ensure_ascii=False, indent=2))
    print('Passou:', registro['passou'], '| divergências:', registro['divergencias'])
print('\nEvidência:', evidencia.name)
print('Resumo:', sum(r['passou'] for r in resultados), '/', len(resultados))
assert all(r['passou'] for r in resultados), 'Há divergências preservadas na evidência; não substituir pelo esperado.'
assert resultado_lcel.status == 'ok' and resultado_chain.status == 'ok'



 F01 Tenho um Gol 2010 1.0 e quero duas velas
{
  "status": "ok",
  "dados": {
    "intencao": "consulta_peca",
    "veiculo": {
      "modelo": "Gol",
      "ano_modelo": 2010,
      "motor": "1.0",
      "combustivel": null
    },
    "peca": "vela",
    "quantidade": 2,
    "cep": null,
    "campos_faltantes": [
      "veiculo.combustivel"
    ]
  },
  "erros": []
}
Passou: True | divergências: []

 F02 Tenho um Gol 2010 1.0 flex e quero duas velas.
{
  "status": "ok",
  "dados": {
    "intencao": "consulta_peca",
    "veiculo": {
      "modelo": "Gol",
      "ano_modelo": 2010,
      "motor": "1.0",
      "combustivel": "flex"
    },
    "peca": "vela",
    "quantidade": 2,
    "cep": null,
    "campos_faltantes": []
  },
  "erros": []
}
Passou: True | divergências: []

 F03 Quero filtro para meu carro.
{
  "status": "erro_validacao",
  "dados": null,
  "erros": [
    "veiculo: Input should be a valid dictionary or instance of VeiculoIdentificado"
  ]
}
Passou: False | divergência

AssertionError: Há divergências preservadas na evidência; não substituir pelo esperado.

## 6. Por que não usar stream para esta entrega?

O consumidor precisa de **JSON completo validado**. Tokens ou objetos parciais podem
não cumprir o schema e não serão apresentados como válidos. Por isso a fase demonstra
`invoke` e `batch`; streaming de estados parciais fica fora deste contrato.

## Limites observados e próximos passos

O schema prova formato/tipos, não fidelidade. Os testes reais são um recorte, não garantia
de qualidade geral. Quantidade zero e CEP inválido aparecem como `erro_validacao`, sem dados
aprovados. O envelope `status/dados/erros` contém o contrato sugerido dentro de `dados`.

Uma consulta de peça por nome solicita configuração ainda ausente; pagamento não solicita
veículo; orçamento solicita quantidade; entrega solicita CEP. Um código de peça explícito
permite simples consulta sem assumir aplicação no veículo. São propostas de coleta, não
validação técnica. O contrato desta fase considera uma peça e a intenção principal.

As fases 2–5 permanecem não implementadas. Catálogo, políticas e casos futuros foram lidos
como contexto, mas não são usados para responder nesta chain.

Implementação com apoio de Codex/OpenAI e revisão de Renan. Referências:
[arquitetura inicial](../01_documentacao/arquitetura_inicial_auto_super_parts_cwb.html),
[decorador chain](https://reference.langchain.com/python/langchain-core/runnables/base/chain),
[runnables/LCEL](https://reference.langchain.com/python/langchain-core/runnables/base).
